In [161]:
# =========================
# Importación de librerías
# =========================

# Redes Neuronales
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Manipulación de datos
import numpy as np
import pandas as pd

# Preprocesamiento y modelado
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# Utilidades
from dataclasses import dataclass


In [162]:
# Carga de Datos desdse DataSet
dataframe = pd.read_csv("dataset_espirometria_interpretacion_5000.csv")
dataframe.head()

,Edad,Genero,Altura_cm,Peso_kg,FVC,FEV1,FEV1_FVC,FVC_pct_pred,FEV1_pct_pred,Post_BD_FVC,Post_BD_FEV1,Post_BD_FEV1_FVC,BD_FEV1_Delta_ml,BD_FEV1_Delta_pct,Fumador,Calidad_Espirometria,Broncodilatador_Realizado,Patron_Espirometrico
0,69,M,164,78.6,3.35,3.00,0.8955,104.8,108.4,3.34,3.04,0.9102,40.0,1.33,0,0,1,Normal
1,75,F,193,125.2,3.53,3.02,0.8555,117.1,92.7,3.53,3.02,0.8555,0.0,0.00,0,1,0,Normal
2,72,F,152,64.9,2.95,1.33,0.4508,91.6,28.8,2.94,1.71,0.5816,380.0,28.57,1,1,1,Obstructivo
3,53,F,189,107.7,2.36,1.89,0.8008,36.4,82.1,2.44,2.30,0.9426,410.0,21.69,0,1,1,Restrictivo
4,79,F,170,53.8,1.73,1.59,0.9191,41.2,80.1,1.87,1.59,0.8503,0.0,0.00,0,1,1,Restrictivo


In [163]:
# Carga de Variables
TARGET_COLUMN = "Patron_Espirometrico"
FEATURE_COLUMNS = [
    "Edad",
    "Genero",
    "Altura_cm",
    "Peso_kg",
    "FVC",
    "FEV1",
    "FEV1_FVC",
    "FVC_pct_pred",
    "FEV1_pct_pred",
    "Post_BD_FVC",
    "Post_BD_FEV1",
    "Post_BD_FEV1_FVC",
    "BD_FEV1_Delta_ml",
    "BD_FEV1_Delta_pct",
    "Fumador",
    "Calidad_Espirometria",
    "Broncodilatador_Realizado",
]
NUMERIC_FEATURES = [
    "Edad",
    "Altura_cm",
    "Peso_kg",
    "FVC",
    "FEV1",
    "FEV1_FVC",
    "FVC_pct_pred",
    "FEV1_pct_pred",
    "Post_BD_FVC",
    "Post_BD_FEV1",
    "Post_BD_FEV1_FVC",
    "BD_FEV1_Delta_ml",
    "BD_FEV1_Delta_pct",
    "Fumador",
    "Calidad_Espirometria",
    "Broncodilatador_Realizado",
]
CATEGORICAL_FEATURES = ["Genero"]
EXPECTED_CLASSES = ["Normal", "Obstructivo", "Restrictivo", "Mixto"]

In [164]:
# Parametros
RANDOM_STATE = 42
TRAIN_SIZE = 0.7
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

In [165]:
# ===============================================
# Separación de predictotres y variable objetivo
# ===============================================

features = dataframe[FEATURE_COLUMNS].copy()
target = dataframe[TARGET_COLUMN].copy()

# =======================================
# División inicial del dataset
# Entrenamiento vs (Validacion + Prueba)
# =======================================
x_train, x_temp, y_train, y_temp = train_test_split(
  features,
  target,
  test_size=(TEST_SIZE + VALIDATION_SIZE),
  random_state=RANDOM_STATE,
  stratify=target,)

In [166]:

# ==========================================
# División secundaria: Validacion vs Prueba
# =========================================

validation_ratio = VALIDATION_SIZE / (TEST_SIZE + VALIDATION_SIZE)
x_val, x_test, y_val, y_test = train_test_split(
  x_temp,
  y_temp,
  test_size=(1 - validation_ratio),
  random_state=RANDOM_STATE,
  stratify=y_temp,)

In [167]:
# Reinicio los índices para tener una numeración consecutiva (0...n) y facilitar el manejo de los datos
x_train=x_train.reset_index(drop=True)
x_val=x_val.reset_index(drop=True)
x_test=x_test.reset_index(drop=True)
y_train=y_train.reset_index(drop=True)
y_val=y_val.reset_index(drop=True)
y_test=y_test.reset_index(drop=True)

In [168]:
@dataclass
class PreparedData:
    x_train: np.ndarray
    x_val: np.ndarray
    x_test: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    preprocessor: Pipeline
    label_encoder: LabelEncoder

In [169]:
def build_preprocessor() -> Pipeline:
    # Estandariza variables numéricas y aplica one-hot encoding a Genero.
    transformer = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ]
    )
    return Pipeline([("transformer", transformer)])

In [172]:
def fit_transform_features(
    x_train: pd.DataFrame,
    x_val: pd.DataFrame,
    x_test: pd.DataFrame,
    y_train: pd.Series,
    y_val: pd.Series,
    y_test: pd.Series,
):
  preprocessor = build_preprocessor()
  label_encoder = LabelEncoder()
  # Ajusta el preprocesado solo con train y reutiliza esa transformación en val/test.
  x_train_processed = preprocessor.fit_transform(x_train)
  x_val_processed = preprocessor.transform(x_val)
  x_test_processed = preprocessor.transform(x_test)
  # Convierte las clases de texto a índices enteros para clasificación multiclase.
  y_train_encoded = label_encoder.fit_transform(y_train)
  y_val_encoded = label_encoder.transform(y_val)
  y_test_encoded = label_encoder.transform(y_test)
  x_train=x_train_processed.astype(np.float32),
  return PreparedData(
        x_train=x_train_processed.astype(np.float32),
        x_val=x_val_processed.astype(np.float32),
        x_test=x_test_processed.astype(np.float32),
        y_train=y_train_encoded.astype(np.int32),
        y_val=y_val_encoded.astype(np.int32),
        y_test=y_test_encoded.astype(np.int32),
        preprocessor=preprocessor,
        label_encoder=label_encoder,
    )

In [173]:
prepared = fit_transform_features(
  x_train,
  x_val,
  x_test,
  y_train,
  y_val,
  y_test,)

In [175]:
def build_baseline_dense(input_dim: int, output_classes: int) -> tf.keras.Model:
    # Red neuronal totalmente conectada para clasificación multiclase.
    model = tf.keras.Sequential(
        [
            # Recibe el vector de features ya preprocesadas.
            tf.keras.layers.Input(shape=(input_dim,)),
            # Capa oculta mínima para baseline inicial.
            tf.keras.layers.Dense(2, activation="relu", name="hidden_dense_2"),
            # Softmax devuelve una probabilidad por cada diagnóstico posible.
            tf.keras.layers.Dense(output_classes, activation="softmax", name="diagnosis_output"),
        ]
    )
     # Define optimizador, función de coste y métrica principal.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [178]:
# Construye la red baseline: 1 capa oculta de 2 neuronas + salida softmax.
model = build_baseline_dense(
  input_dim=prepared.x_train.shape[1],
  output_classes=len(prepared.label_encoder.classes_),
)

In [179]:
# Detiene el entrenamiento si la pérdida de validación deja de mejorar.
early_stopping = tf.keras.callbacks.EarlyStopping(
monitor="val_loss",
patience=10,
restore_best_weights=True,
)


In [181]:
verbose: int = 1
# Entrena el modelo sobre train y monitorea desempeño en validation.
history = model.fit(
prepared.x_train,
prepared.y_train,
validation_data=(prepared.x_val, prepared.y_val),
epochs=100,
batch_size=32,
callbacks=[early_stopping],
verbose=verbose,
)


Epoch 1/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2406 - loss: 1.4396 - val_accuracy: 0.3200 - val_loss: 1.3506
Epoch 2/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4369 - loss: 1.2878 - val_accuracy: 0.5027 - val_loss: 1.2359
Epoch 3/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5226 - loss: 1.1881 - val_accuracy: 0.5427 - val_loss: 1.1428
Epoch 4/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5600 - loss: 1.0953 - val_accuracy: 0.6000 - val_loss: 1.0441
Epoch 5/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6140 - loss: 0.9894 - val_accuracy: 0.6573 - val_loss: 0.9359
Epoch 6/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6534 - loss: 0.8818 - val_accuracy: 0.6773 - val_loss: 0.8337
Epoch 7/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7531 - loss: 0.7834 - val_accuracy: 0.8600 - val_loss: 0.7420
Epoch 8/100
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8923 - loss: 0.6913 - val_accu

In [209]:
# Datos de Prueba
datos_prueba = {
    "Edad": 61,
    "Genero": "M",
    "Altura_cm": 171,
    "Peso_kg": 91,
    "FVC": 1,
    "FEV1": 1,
    "FVC_pct_pred": 60,
    "FEV1_pct_pred": 98.4,
    "Post_BD_FVC": 0,
    "Post_BD_FEV1": 0,
    "Fumador": 1,
    "Calidad_Espirometria": 1
}




In [210]:
# Calcula campos derivados usados como predictores:
# FEV1_FVC, Post_BD_FEV1_FVC, BD_FEV1_Delta_ml y BD_FEV1_Delta_pct
def calcular_campos_derivados(datos):
    datos = datos.copy()

    datos["Edad"] = float(datos["Edad"])
    datos["Altura_cm"] = float(datos["Altura_cm"])
    datos["Peso_kg"] = float(datos["Peso_kg"])
    datos["FVC"] = float(datos["FVC"])
    datos["FEV1"] = float(datos["FEV1"])
    datos["FVC_pct_pred"] = float(datos["FVC_pct_pred"])
    datos["FEV1_pct_pred"] = float(datos["FEV1_pct_pred"])
    datos["Fumador"] = int(datos.get("Fumador", 0))
    datos["Calidad_Espirometria"] = int(datos.get("Calidad_Espirometria", 1))

    if datos["FVC"] <= 0:
        raise ValueError("FVC debe ser mayor a cero.")

    if datos["FEV1"] <= 0:
        raise ValueError("FEV1 debe ser mayor a cero.")

    post_fvc = datos.get("Post_BD_FVC")
    post_fev1 = datos.get("Post_BD_FEV1")

    sin_broncodilatador = (
        post_fvc in ("", None, 0, 0.0)
        and post_fev1 in ("", None, 0, 0.0)
    )

    if sin_broncodilatador:
        datos["Post_BD_FVC"] = datos["FVC"]
        datos["Post_BD_FEV1"] = datos["FEV1"]
        datos["Broncodilatador_Realizado"] = 0
    else:
        datos["Post_BD_FVC"] = float(post_fvc) if post_fvc not in ("", None, 0, 0.0) else datos["FVC"]
        datos["Post_BD_FEV1"] = float(post_fev1) if post_fev1 not in ("", None, 0, 0.0) else datos["FEV1"]
        datos["Broncodilatador_Realizado"] = 1

    if datos["Post_BD_FVC"] <= 0:
        raise ValueError("Post_BD_FVC debe ser mayor a cero.")

    datos["FEV1_FVC"] = round(datos["FEV1"] / datos["FVC"], 4)

    datos["Post_BD_FEV1_FVC"] = round(
        datos["Post_BD_FEV1"] / datos["Post_BD_FVC"],
        4
    )

    datos["BD_FEV1_Delta_ml"] = round(
        (datos["Post_BD_FEV1"] - datos["FEV1"]) * 1000,
        1
    )

    datos["BD_FEV1_Delta_pct"] = round(
        ((datos["Post_BD_FEV1"] - datos["FEV1"]) / datos["FEV1"]) * 100,
        2
    )

    return datos


datos_completos = calcular_campos_derivados(datos_prueba)

In [207]:
print(datos_completos)

{'Edad': 61.0, 'Genero': 'M', 'Altura_cm': 171.0, 'Peso_kg': 91.0, 'FVC': 3.0, 'FEV1': 1.0, 'FVC_pct_pred': 60.0, 'FEV1_pct_pred': 98.4, 'Post_BD_FVC': 3.0, 'Post_BD_FEV1': 1.0, 'Fumador': 1, 'Calidad_Espirometria': 1, 'Broncodilatador_Realizado': 0, 'FEV1_FVC': 0.3333, 'Post_BD_FEV1_FVC': 0.3333, 'BD_FEV1_Delta_ml': 0.0, 'BD_FEV1_Delta_pct': 0.0}


In [211]:
# Convierte datos_completos a DataFrame
X_input = pd.DataFrame([datos_completos])
# Aplica el mismo preprocesamiento usado durante el entrenamiento
X_processed = prepared.preprocessor.transform(X_input).astype(np.float32)

# Hago prediccion
proba = model.predict(X_processed)

pred_num = np.argmax(proba, axis=1)

pred_class = prepared.label_encoder.inverse_transform(pred_num)
for clase, prob in zip(prepared.label_encoder.classes_, proba[0]):
    print(f"{clase}: {prob:.2%}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Mixto: 0.10%
Normal: 0.00%
Obstructivo: 0.00%
Restrictivo: 99.90%


In [212]:
print(pred_class)
print(proba)
print(prepared.label_encoder.classes_)

['Restrictivo']
[[1.0032218e-03 6.9150366e-11 2.7619027e-12 9.9899679e-01]]
['Mixto' 'Normal' 'Obstructivo' 'Restrictivo']
